# 1. Import Library dan Setup
Bagian ini digunakan untuk mengimpor semua library yang dibutuhkan untuk proses pemodelan, seperti `scikit-learn` untuk preprocessing/split, `xgboost`, dan `tensorflow.keras` untuk deep learning.

In [ ]:
import numpy as np
import scipy.sparse as sp
import joblib
import os
import tensorflow as tf
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, GlobalMaxPooling1D, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

# 2. Load Extracted Features dan Labels
Membaca fitur-fitur yang telah diekstrak pada tahapan sebelumnya (TF-IDF, Word2Vec, GloVe, dan FastText) beserta target labels dari folder `extracted_features/`.

In [ ]:
# Load data
feature_dir = 'extracted_features'

# Memuat label
y = np.load(os.path.join(feature_dir, 'y_labels.npy'))

# Memuat semua fitur
X_tfidf = sp.load_npz(os.path.join(feature_dir, 'X_tfidf.npz')).todense() # Convert sparse to dense array
X_w2v = np.load(os.path.join(feature_dir, 'X_w2v.npy'))
X_glove = np.load(os.path.join(feature_dir, 'X_glove.npy'))
X_ft = np.load(os.path.join(feature_dir, 'X_ft.npy'))

print(f"Bentuk TF-IDF    : {X_tfidf.shape}")
print(f"Bentuk Word2Vec  : {X_w2v.shape}")
print(f"Bentuk GloVe     : {X_glove.shape}")
print(f"Bentuk FastText  : {X_ft.shape}")
print(f"Bentuk Label     : {y.shape}")

# 3. Split Data (Train & Test)
Membagi masing-masing set fitur menjadi data latih (80%) dan data uji (20%) menggunakan parameter `random_state` yang sama agar pembagian observasinya konsisten pada setiap eksperimen.

In [ ]:
# Kita akan membuat dictionary untuk memudahkan iterasi
features_dict = {
    'TF-IDF': np.asarray(X_tfidf),
    'Word2Vec': X_w2v,
    'GloVe': X_glove,
    'FastText': X_ft
}

data_splits = {}

for name, feature_matrix in features_dict.items():
    X_train, X_test, y_train, y_test = train_test_split(feature_matrix, y, test_size=0.2, random_state=42)
    data_splits[name] = {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }
    print(f"[{name}] Train: {X_train.shape}, Test: {X_test.shape}")

# 4. Fungsi Evaluasi Model
Mendefinisikan fungsi pembantu untuk menyimpan log performa, agar nantinya dapat digunakan pada evaluasi akhir keseluruhan model.

In [ ]:
results = []

def evaluate_model(model_name, feature_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    results.append({
        'Model': model_name,
        'Feature': feature_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })
    
    print(f"\n[{model_name} | {feature_name}]")
    print(f"Accuracy  : {acc:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")

# 5. Pemodelan dengan XGBoost
Algoritma berbasis tree-ensemble ini cukup handal dalam klasifikasi. XGBoost akan dilatih dengan keempat varian representasi teks.

In [ ]:
for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict(X_test)
    
    evaluate_model('XGBoost', feature_name, y_test, y_pred)

# 6. Pemodelan dengan CNN (Convolutional Neural Network)
CNN dapat menangkap relasi spasial dalam sequence data. Untuk Deep Learning (CNN dan LSTM) yang mensyaratkan input sequence 3D `(batch_size, sequence_length, embedding_dim)`, kita akan mereshape matriks fitur teks (berdimensi `N x Dimensi_Fitur`) menjadi sekumpulan sequence.

In [ ]:
def create_cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, padding='same', activation='relu', input_shape=input_shape),
        GlobalMaxPooling1D(),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    # Reshape (samples, features, 1) agar bisa masuk ke layer Conv1D
    X_train_cnn = np.expand_dims(X_train, axis=2)
    X_test_cnn = np.expand_dims(X_test, axis=2)
    
    model = create_cnn_model((X_train_cnn.shape[1], 1))
    
    # Gunakan verbosity 0 untuk mencegah output yang terlalu panjang
    model.fit(X_train_cnn, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_cnn, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('CNN', feature_name, y_test, y_pred)

# 7. Pemodelan dengan LSTM (Long Short-Term Memory)
LSTM sangat tangguh dalam menangkap dependensi jangka panjang pada sequence data. Kita akan memperlakukan fitur seperti sequence time-step agar cocok dengan layer LSTM.

In [ ]:
def create_lstm_model(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=False, input_shape=input_shape),
        Dropout(0.5),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

for feature_name in features_dict.keys():
    X_train = data_splits[feature_name]['X_train']
    X_test = data_splits[feature_name]['X_test']
    y_train = data_splits[feature_name]['y_train']
    y_test = data_splits[feature_name]['y_test']
    
    # Reshape (samples, 1, features)
    X_train_lstm = np.expand_dims(X_train, axis=1)
    X_test_lstm = np.expand_dims(X_test, axis=1)
    
    model = create_lstm_model((1, X_train.shape[1]))
    
    model.fit(X_train_lstm, y_train, epochs=10, batch_size=32, 
              validation_split=0.2, callbacks=[early_stopping], verbose=0)
    
    y_pred_probs = model.predict(X_test_lstm, verbose=0)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()
    
    evaluate_model('LSTM', feature_name, y_test, y_pred)

# 8. Evaluasi dan Perbandingan Keseluruhan
Menyusun semua hasil eksperimen ke dalam Pandas DataFrame untuk membandingkan secara eksplisit performa masing-masing model yang dikombinasikan dengan jenis ekstraksi fitur yang berbeda.

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['F1-Score', 'Accuracy'], ascending=False).reset_index(drop=True)

print("=== PERBANDINGAN MODEL DAN FEATURE EXTRACTION ===")
display(results_df)

# 9. Menyimpan Model Terbaik
Dari hasil perbandingan di atas, model beserta jenis representasi vektornya yang memberikan F1-Score atau Accuracy terbaik akan disimpan di dalam direktori `Model/` untuk dapat digunakan pada saat deployment (Gradio / Streamlit).

In [ ]:
# Pastikan folder model ada
os.makedirs('../Model', exist_ok=True)

best_model_idx = results_df.index[0]
best_model_name = results_df.loc[best_model_idx, 'Model']
best_feature_name = results_df.loc[best_model_idx, 'Feature']

print(f"Model terbaik yang akan disimpan adalah: {best_model_name} dengan fitur {best_feature_name}")

# PENTING:
# Di dunia nyata, Anda harus melatih ulang model tersebut pada KESELURUHAN DATA
# atau menyimpan instansi model keras/sklearn yang mendapatkan hasil terbaik tadi.
# Snippet di bawah ini adalah contoh re-train dan save menggunakan XGBoost.

# Jika yang terbaik adalah XGBoost:
if best_model_name == 'XGBoost':
    print("Menyimpan model XGBoost...")
    best_xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    best_xgb.fit(data_splits[best_feature_name]['X_train'], data_splits[best_feature_name]['y_train'])
    joblib.dump(best_xgb, '../Model/best_fake_news_model.joblib')
    print("Model disimpan ke '../Model/best_fake_news_model.joblib'")

# Jika Deep Learning, gunakan model.save()
elif best_model_name in ['CNN', 'LSTM']:
    print(f"Menyimpan model {best_model_name}...")
    # Latih model final (Contoh singkat, sebaiknya gunakan instance terbaik yang dihasilkan loop atas)
    if best_model_name == 'CNN':
        model_final = create_cnn_model((data_splits[best_feature_name]['X_train'].shape[1], 1))
        X_train_final = np.expand_dims(data_splits[best_feature_name]['X_train'], axis=2)
    else:
        model_final = create_lstm_model((1, data_splits[best_feature_name]['X_train'].shape[1]))
        X_train_final = np.expand_dims(data_splits[best_feature_name]['X_train'], axis=1)
        
    model_final.fit(X_train_final, data_splits[best_feature_name]['y_train'], epochs=5, batch_size=32, verbose=0)
    model_final.save('../Model/best_fake_news_model.keras')
    print("Model disimpan ke '../Model/best_fake_news_model.keras'")
